In [0]:
%sql

USE CATALOG mvp;
USE SCHEMA gold;
CREATE OR REPLACE TABLE gold_market_share_setor_pais 
USING delta
AS 
WITH importacoes_por_setor_ano AS (
    
    SELECT 
        co_ano,
        setor_capitulo,
        SUM(vl_fob) AS vl_fob_total_setor
    FROM mvp.silver.comex_importacao_limpa
    GROUP BY co_ano, setor_capitulo
),
importacoes_por_pais_setor_ano AS (
    SELECT 
        co_ano,
        setor_capitulo,
        co_pais,
        SUM(vl_fob) AS vl_fob_pais_setor
    FROM mvp.silver.comex_importacao_limpa
    GROUP BY co_ano, setor_capitulo, co_pais
)
SELECT 
    p.co_ano,
    p.setor_capitulo,
    p.co_pais,
    p.vl_fob_pais_setor,
    t.vl_fob_total_setor,
    
    ROUND((p.vl_fob_pais_setor / t.vl_fob_total_setor) * 100, 4) AS market_share_perc
FROM importacoes_por_pais_setor_ano p
JOIN importacoes_por_setor_ano t
  ON p.co_ano = t.co_ano 
  AND p.setor_capitulo = t.setor_capitulo;

In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA gold;

SELECT COUNT(*) AS total_linhas_gold FROM gold_market_share_setor_pais;

SELECT 
    co_ano,
    setor_capitulo,
    co_pais,
    vl_fob_pais_setor,
    vl_fob_total_setor,
    market_share_perc
FROM gold_market_share_setor_pais
WHERE co_ano = 2024
ORDER BY market_share_perc DESC
LIMIT 10;